# 02 — Data quality and exploratory analysis

**Objectives**

- Generate data from a versioned, deterministic source.
- Check schema, identity uniqueness, label meaning, missingness, and prevalence.
- Inspect training/validation cohort change without opening frozen-test labels.

**Prerequisite:** lesson 01.


In [ ]:
import pandas as pd

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split, prepare_dataset, validate_dataset
from aai_local_classification.tracking import local_paths
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
paths = local_paths(root)
manifest = prepare_dataset(settings, paths.data_root)
pd.DataFrame([item.model_dump(mode="json") for item in manifest.artifacts])


The frozen-test file is hashed and its dates/row count are known, but its label
rate is deliberately withheld from the manifest until release evaluation. A
digest detects change; it does not preserve the raw dataset for you.


In [ ]:
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
pd.DataFrame(
    {
        "train": validate_dataset(train, settings),
        "validation": validate_dataset(validation, settings),
    }
)


In [ ]:
missingness = pd.concat(
    {
        "train": train[list(settings.features.model_columns)].isna().mean(),
        "validation": validation[list(settings.features.model_columns)].isna().mean(),
    },
    axis=1,
)
missingness.sort_values("train", ascending=False)


In [ ]:
development = pd.concat([train, validation], ignore_index=True)
monthly = development.groupby(development.snapshot_date.dt.to_period("M")).agg(
    rows=(settings.data.target_column, "size"),
    positive_rate=(settings.data.target_column, "mean"),
    average_fee=("monthly_fee", "mean"),
)
monthly


### Exercise

Add one check that should fail before training if a category suddenly dominates
validation. Decide whether it is a hard failure or a warning and explain why.

**Hint:** compare normalized value counts, but remember that distribution change
can be legitimate production behavior rather than corrupt data.

**Checkpoint:** the generator is deterministic, feature missingness is within
contract, IDs are unique, both labels exist, and only train/validation labels
have been inspected.

Next: **03_leakage_safe_splits.ipynb**.
